# 07 — xc 上线后监控与复盘(runbook 第 8 步)

上线后定期跑三件事(内核 env_ml,需 xgboost):

1. **分数漂移**:新批次校准分分布 vs 参照窗(训练表 OOT 边界前的样本重放打分),PSI 量化;
2. **特征漂移**:两个 bundle 头部重要特征在新批次 vs 参照窗的 PSI(缺失率跳变也会推高);
3. **转化复盘**:标签成熟后,用重新合表的 `xc_full.csv` 在上线窗口按**线上 α** 复算 top-K 各段转化,对照验收报告。

漂移明显或提升衰减 → 回 runbook 第 1 步用新数据重训(新 run-id,旧 bundle 可回滚);特征上游口径变更 → 重走第 2~6 步。更细的曲线/分段分析用 06(`ALPHA` 填线上 `fusion_spec.json` 的值、`START_DT` 填上线日)。

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from predict_template import Predictor
from wdm.analysis.psi import compute_psi, flag
from wdm.config import load_config
from wdm.model.fusion import fuse
from wdm.utils.paths import model_run_dir

# ---- 参数:与线上一致的 bundle / 融合 spec;新批次 = 线上待打分原始特征 ----
RESP_PRODUCT, RESP_RUN_ID = 'xc_resp_finish', 'prod_20260610'
QUAL_PRODUCT, QUAL_RUN_ID = 'xc_qual_finish', 'prod_20260610'
QUAL_STAGE = 'is_credit_succ'    # 资质 V2 上线则换 'is_credit_1v1'(QUAL_PRODUCT 同步换 1v1)
FUSION_SPEC_PATH = None          # None = 按上面四项推默认验收目录;或手填路径
NEW_BATCH = 'data/xc_scoring_batch.csv'   # 相对仓库根;列同训练特征,允许缺失
REF_SAMPLE_N = 200000            # 参照窗抽样上限(控内存/耗时)

cfgs = {'resp': load_config(RESP_PRODUCT), 'qual': load_config(QUAL_PRODUCT)}
repo = Path(cfgs['resp']['_repo_root'])
bundles = {'resp': model_run_dir(cfgs['resp'], RESP_RUN_ID),
           'qual': model_run_dir(cfgs['qual'], QUAL_RUN_ID)}
preds = {n: Predictor(str(b)) for n, b in bundles.items()}
for n, p in preds.items():
    if not p.has_calibration:
        print('警告:{0} bundle 无 calibration.json,本监控退化为原始分口径'.format(n))

if FUSION_SPEC_PATH is None:
    FUSION_SPEC_PATH = (repo / 'artifacts' / 'funnel_eval' /
                        '{0}-{1}__{2}-{3}'.format(RESP_PRODUCT, RESP_RUN_ID,
                                                  QUAL_PRODUCT, QUAL_RUN_ID) /
                        'fusion_spec.json')
spec_path = Path(FUSION_SPEC_PATH)
if spec_path.is_file():
    spec = json.loads(spec_path.read_text(encoding='utf-8'))
    ALPHA = float(spec['alpha'])
    print('alpha={0} ({1}) <- {2}'.format(ALPHA, spec.get('alpha_source'), spec_path))
else:
    spec, ALPHA = None, 0.5
    print('警告:未找到 fusion_spec.json({0}),alpha 回退 0.5 — '
          '复盘必须用线上实际投放的 α!'.format(spec_path))

In [ ]:
# 参照窗 = 训练表上 OOT 边界之前的样本(模型训练/调参见过的分布),抽样重放打分作漂移基准;
# 另对照 bundle 内 validation_samples.csv 的预期分数区间(100 行,粗校验)
manifest = json.loads((bundles['resp'] / 'run_manifest.json').read_text(encoding='utf-8'))
oot_min = (manifest.get('split_boundaries') or {}).get('oot_min_dt')
time_col = cfgs['resp']['data']['time_column']
needed = sorted({time_col} | set(preds['resp'].base_features) | set(preds['qual'].base_features))
full = pd.read_csv(repo / 'data' / 'xc_full.csv', usecols=needed)
dtv = pd.to_numeric(full[time_col], errors='coerce')
ref = full[dtv < oot_min] if oot_min is not None else full
if len(ref) > REF_SAMPLE_N:
    ref = ref.sample(REF_SAMPLE_N, random_state=42)
print('reference rows:', len(ref), '| dt <', oot_min)

ref_scores = {}
for n, p in preds.items():
    raw = np.asarray(p.predict_proba(ref[p.base_features]), dtype=float)
    ref_scores[n] = p.calibrate(raw) if p.has_calibration else raw
ref_scores['fused'] = fuse(ref_scores['resp'], ref_scores['qual'], ALPHA)

vs = pd.read_csv(bundles['resp'] / 'validation_samples.csv')
col = ('y_pred_calibrated_expected' if 'y_pred_calibrated_expected' in vs.columns
       else 'y_pred_expected')
print('resp validation_samples {0} 区间: [{1:.6f}, {2:.6f}]'.format(
    col, vs[col].min(), vs[col].max()))
pd.DataFrame({n: pd.Series(s).describe() for n, s in ref_scores.items()}).round(6)

In [ ]:
# 新批次打分 + 分数 PSI(expected=参照窗):psi >= 0.10 shift / >= 0.25 broken
# → 排查上游特征口径与人群结构;fused 分位数表给投放 cutoff 参考
new = None
nb_path = repo / NEW_BATCH
if nb_path.is_file():
    new = pd.read_csv(nb_path)
    print('new batch rows:', len(new))
    new_scores = {}
    for n, p in preds.items():
        raw = np.asarray(p.predict_proba(new[p.base_features]), dtype=float)
        new_scores[n] = p.calibrate(raw) if p.has_calibration else raw
    new_scores['fused'] = fuse(new_scores['resp'], new_scores['qual'], ALPHA)

    rows = []
    for n in ['resp', 'qual', 'fused']:
        v = float(compute_psi(ref_scores[n], new_scores[n], n_bins=10))
        rows.append({'score': n, 'psi': round(v, 4), 'flag': flag(v),
                     'ref_mean': round(float(np.mean(ref_scores[n])), 6),
                     'new_mean': round(float(np.mean(new_scores[n])), 6)})
    display(pd.DataFrame(rows))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    for ax, n in zip(axes, ['resp', 'qual', 'fused']):
        ax.hist(ref_scores[n], bins=40, alpha=0.5, density=True, label='reference')
        ax.hist(new_scores[n], bins=40, alpha=0.5, density=True, label='new batch')
        ax.set_title(n); ax.legend()
    plt.tight_layout()

    qs = [0.50, 0.80, 0.90, 0.95, 0.99]
    display(pd.DataFrame({'quantile': qs,
                          'fused_cutoff': np.quantile(new_scores['fused'], qs)}).round(6))
else:
    print('新批次文件不存在:{0}'.format(nb_path))
    print('把线上 raw 特征 CSV 放到该路径(或改 NEW_BATCH)后重跑本格;下面两格依赖它')

In [ ]:
# 头部重要特征 PSI(参照窗 vs 新批次)。compute_psi 为 NaN-aware,缺失单独成桶,
# 缺失率跳变同样会推高 PSI。broken 多 → 上游口径变更?须重走 runbook 第 2~6 步
if new is not None:
    TOP_N_FEAT = 20
    feats = []
    for n in ['resp', 'qual']:
        impf = bundles[n] / 'importance.csv'
        if impf.is_file():
            feats += pd.read_csv(impf)['feature'].head(TOP_N_FEAT).tolist()
    feats = [f for f in dict.fromkeys(feats) if f in new.columns and f in ref.columns]
    rows = []
    for f in feats:
        rv = pd.to_numeric(ref[f], errors='coerce').values
        nv = pd.to_numeric(new[f], errors='coerce').values
        v = float(compute_psi(rv, nv, n_bins=10))
        rows.append({'feature': f, 'psi': round(v, 4), 'flag': flag(v),
                     'ref_missing': round(float(np.isnan(rv).mean()), 4),
                     'new_missing': round(float(np.isnan(nv).mean()), 4)})
    fd = pd.DataFrame(rows).sort_values('psi', ascending=False)
    display(fd)
    n_bad = int((fd['flag'] != 'stable').sum())
    if n_bad:
        print('{0} 个头部特征漂移(shift/broken)→ 逐个排查上游取数口径'.format(n_bad))
else:
    print('先在上一格加载新批次')

In [ ]:
# 转化复盘(标签成熟后):重跑 build_xc_dataset.py 得到含上线窗口的新 xc_full.csv,
# 设 LAUNCH_DT 后本格按线上 α 固定融合排序,复算 top-K 各段转化与 lift,对照验收报告。
# 注意:α 必须用线上 spec 的值(在新窗口重拟合评估的不是实际投放的那套排序)
LAUNCH_DT = None   # 形如 20260701;None 跳过
K = 0.10           # 与实际投放比例一致
if LAUNCH_DT:
    stages = ['is_reg', 'is_finish_task', QUAL_STAGE]
    cols = sorted(set(stages) | {time_col} |
                  set(preds['resp'].base_features) | set(preds['qual'].base_features))
    post = pd.read_csv(repo / 'data' / 'xc_full.csv', usecols=cols)
    dtp = pd.to_numeric(post[time_col], errors='coerce')
    post = post[dtp >= LAUNCH_DT].reset_index(drop=True)
    print('post-launch rows:', len(post), '| dt >=', LAUNCH_DT)
    sc = {}
    for n, p in preds.items():
        raw = np.asarray(p.predict_proba(post[p.base_features]), dtype=float)
        sc[n] = p.calibrate(raw) if p.has_calibration else raw
    fused = fuse(sc['resp'], sc['qual'], ALPHA)
    n_top = max(1, int(np.ceil(K * len(post))))
    top = np.zeros(len(post), dtype=bool)
    top[np.argsort(-fused, kind='stable')[:n_top]] = True
    rows = []
    for s in stages:
        fl = (pd.to_numeric(post[s], errors='coerce').fillna(0) > 0).astype(int).values
        base, topr = float(fl.mean()), float(fl[top].mean())
        rows.append({'stage': s, 'base_rate': round(base, 4), 'topK_rate': round(topr, 4),
                     'lift': round(topr / base, 3) if base > 0 else np.nan})
    print('上线窗口实际(fused@alpha={0:.2f}, top {1:.0%}):'.format(ALPHA, K))
    display(pd.DataFrame(rows))
    acc = spec_path.parent / 'funnel_eval.csv'
    if acc.is_file():
        a = pd.read_csv(acc)
        a = a[(a['view'] == 'absolute') & (a['score'] == 'fused_alpha')
              & np.isclose(a['top_k_pct'], K) & a['stage'].isin(stages)]
        print('验收报告对应行(上线前 OOT 口径,作对照):')
        display(a[['stage', 'base_rate', 'topk_rate', 'lift']].round(4))
else:
    print('LAUNCH_DT 未设置 — 投放且标签成熟、重新合表后再跑本格复盘')